# SupremeAI - Gemma 2B Inference Server

Runs Gemma 2B-it on Colab and serves OpenAI-compatible API for SupremeAI.

### Requirements:
1. Runtime > Change runtime type > T4 GPU
2. HuggingFace Token - https://huggingface.co/settings/tokens
3. ngrok Token - https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Step 1: Install dependencies
!pip uninstall -y transformers optimum accelerate bitsandbytes airllm -q 2>/dev/null
!pip install -q transformers accelerate bitsandbytes
!pip install -q optimum
!pip install -q airllm
!pip install -q flask flask-cors pyngrok

import torch
import sys
print('Installed packages:')
print(f'  Python: {sys.version.split()[0]}')
print(f'  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ NO GPU - Change Runtime to T4!')

In [ ]:
# Step 2: Configure tokens - Gemma 2B Model (Token Set 1)
HF_TOKEN = 'hf_PLACEHOLDER_PASTE_YOUR_TOKEN'
NGROK_TOKEN = '3C1Nii9tdcVN5pPSUB2jwYEGxPT_HFmSpXkFq7uBYRs7VvLR'

MODEL_ID = 'google/gemma-2b-it'
COMPRESSION = '4bit'
MAX_NEW_TOKENS = 512
PORT = 8081

print(f"✓ Model Configuration (Token Set 1):")
print(f"  Model: Gemma 2B-it")
print(f"  Model ID: {MODEL_ID}")
print(f"  Compression: {COMPRESSION}")
print(f"  Max Tokens: {MAX_NEW_TOKENS}")

In [ ]:
# Step 3: Load Gemma 2B model
import time

print('Loading Gemma 2B model...')
start = time.time()

try:
    from airllm import AutoModel
    kw = {'compression': COMPRESSION}
    if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10:
        kw['hf_token'] = HF_TOKEN
    
    print('  Downloading from HuggingFace...')
    model = AutoModel.from_pretrained(MODEL_ID, **kw)
    print(f'✓ Model loaded in {time.time() - start:.0f}s')
    
except Exception as e:
    print(f'✗ AirLLM failed: {str(e)}')
    print('  Attempting fallback with transformers...')
    
    from transformers import AutoModelForCausalLM, AutoTokenizer
    token = HF_TOKEN if HF_TOKEN.startswith('hf_') else None
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=token,
        device_map='auto',
        load_in_4bit=True if COMPRESSION == '4bit' else False
    )
    model.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    print(f'✓ Model loaded (transformers) in {time.time() - start:.0f}s')

print(f'\nModel ready: {MODEL_ID}')

In [ ]:
# Step 4: Start API Server + ngrok
import uuid, time, torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)
stats = {'req': 0, 'err': 0, 'start': time.time()}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'provider': 'airllm-colab',
        'model_active': MODEL_ID,
        'model_name': 'Gemma 2B-it',
        'compression': COMPRESSION,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'uptime': int(time.time() - stats['start']),
        'requests': stats['req'],
        'errors': stats['err']
    })

@app.route('/v1/models', methods=['GET'])
def list_models():
    return jsonify({
        'object': 'list',
        'data': [{
            'id': MODEL_ID,
            'object': 'model',
            'owned_by': 'airllm-colab',
            'name': 'Gemma 2B-it'
        }]
    })

@app.route('/v1/chat/completions', methods=['POST'])
def chat():
    try:
        stats['req'] += 1
        data = request.json
        msgs = data.get('messages', [])
        mt = min(data.get('max_tokens', MAX_NEW_TOKENS), MAX_NEW_TOKENS)
        
        parts = []
        for m in msgs:
            r, c = m.get('role', 'user'), m.get('content', '')
            if r == 'system':
                parts.append('[INST] <<SYS>>\n' + c + '\n<</SYS>>\n')
            elif r == 'user':
                parts.append('[INST] ' + c + ' [/INST]')
            elif r == 'assistant':
                parts.append(c)
        
        prompt = '\n'.join(parts)
        
        toks = model.tokenizer(
            [prompt],
            return_tensors='pt',
            return_attention_mask=False,
            truncation=True,
            max_length=2048,
            padding=False
        )
        il = len(toks['input_ids'][0])
        gen = model.generate(
            toks['input_ids'].cuda(),
            max_new_tokens=mt,
            use_cache=True,
            return_dict_in_generate=True
        )
        nt = gen.sequences[0][il:]
        txt = model.tokenizer.decode(nt, skip_special_tokens=True).strip()
        
        return jsonify({
            'id': 'chatcmpl-' + uuid.uuid4().hex[:12],
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': MODEL_ID,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': txt},
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': il,
                'completion_tokens': len(nt),
                'total_tokens': il + len(nt)
            }
        })
    except Exception as e:
        stats['err'] += 1
        return jsonify({
            'error': {
                'message': str(e),
                'type': 'server_error',
                'code': 500
            }
        }), 500

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
url = tunnel.public_url

print('\n' + '='*70)
print('🚀 SupremeAI Gemma 2B Server LIVE!')
print('='*70)
print(f'Model: Gemma 2B-it ({MODEL_ID})')
print(f'Compression: {COMPRESSION} | Max Tokens: {MAX_NEW_TOKENS}')
print('='*70)
print(f'Public URL:    {url}')
print(f'Chat Endpoint: {url}/v1/chat/completions')
print(f'Health Check:  {url}/health')
print(f'List Models:   {url}/v1/models')
print('='*70)
print('Configure in SupremeAI backend:')
print(f'  AIRLLM_ENDPOINT={url}/v1/chat/completions')
print(f'  AIRLLM_HEALTHCHECK_URL={url}/health')
print('='*70)
print('⚠️  Keep this cell running - do not stop!')
app.run(host='0.0.0.0', port=PORT)